# Segmentação de pluma de fumaça com o SAM (Segment Anything Model)

**Objetivo:** segmentar uma pluma de fumaça em uma imagem, avaliar criticamente o resultado do SAM e melhorar código, prompts e apresentação da máscara.

**Roteiro do notebook**
1. Preparação (instalação, modelo, imagem)
2. Observação inicial: o que o SAM faz sem orientação? (baseline)
3. Três configurações de prompt (caixa, pontos positivos, caixa + positivos + negativos), lado a lado: imagem original | máscara binária | sobreposição
4. Melhorias: pós-processamento, refinamento iterativo e pré-processamento de contraste
5. Métricas comparativas
6. Análise crítica (características que dificultaram a segmentação)
7. Exportação da máscara final

> Use GPU: *Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)*.

## 1. Preparação

In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git opencv-python-headless matplotlib pandas scipy
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

MODEL_TYPE = "vit_b"   # vit_b (leve), vit_l ou vit_h (mais preciso, mas pesado)
CHECKPOINT = "sam_vit_b_01ec64.pth"
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT).to(device)
predictor = SamPredictor(sam)

### Carregar a imagem
O notebook tenta baixar `pluma.png` do repositório do GitHub. Se falhar, ele pede o upload manual da imagem (JPG/PNG).

In [ ]:
import os
from google.colab import files
IMG_URL = "https://raw.githubusercontent.com/Gian3010/Detec-o-e-segmenta-o-de-uma-pluma-com-SAM/main/pluma.png"
if not os.path.exists("pluma.png"):
    os.system(f"wget -q -O pluma.png {IMG_URL}")
if os.path.exists("pluma.png") and os.path.getsize("pluma.png") > 1000:
    path = "pluma.png"
else:
    up = files.upload()
    path = list(up.keys())[0]

image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
MAX_SIDE = 1600
h, w = image.shape[:2]
if max(h, w) > MAX_SIDE:
    s = MAX_SIDE / max(h, w)
    image = cv2.resize(image, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
print("Tamanho da imagem (altura, largura):", image.shape[:2])

### Grade de coordenadas
Use a grade para escolher a caixa e os pontos (coordenadas em pixels: x = horizontal, y = vertical).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)
ax.set_xticks(range(0, image.shape[1], 100))
ax.set_yticks(range(0, image.shape[0], 100))
ax.grid(color="yellow", alpha=0.4, linewidth=0.6)
ax.tick_params(labelsize=8)
ax.set_title("Imagem com grade (x, y em pixels)")
plt.show()

## 2. Funções auxiliares de visualização

In [ ]:
def overlay(img, mask, color=(255, 0, 0), alpha=0.5, contour=True):
    out = img.copy().astype(np.float32)
    c = np.array(color, dtype=np.float32)
    out[mask] = (1 - alpha) * out[mask] + alpha * c
    out = out.astype(np.uint8)
    if contour:
        cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cs, -1, (255, 255, 0), 2)
    return out

def draw_prompts(ax, box=None, pos=None, neg=None):
    if box is not None:
        x0, y0, x1, y1 = box
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="lime", linewidth=2))
    if pos is not None and len(pos):
        p = np.array(pos); ax.scatter(p[:,0], p[:,1], c="lime", marker="*", s=250, edgecolor="white", linewidth=1.2)
    if neg is not None and len(neg):
        n = np.array(neg); ax.scatter(n[:,0], n[:,1], c="red", marker="X", s=180, edgecolor="white", linewidth=1.2)

def side_by_side(img, mask, title="", box=None, pos=None, neg=None, score=None):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    axs[0].imshow(img); draw_prompts(axs[0], box, pos, neg)
    axs[0].set_title("Imagem original (+ prompts)")
    axs[1].imshow(mask, cmap="gray"); axs[1].set_title("Máscara binária")
    axs[2].imshow(overlay(img, mask)); axs[2].set_title("Máscara sobreposta")
    for a in axs: a.axis("off")
    t = title + (f"  |  score SAM = {score:.3f}" if score is not None else "")
    fig.suptitle(t + f"  |  área = {100*mask.mean():.1f}% da imagem", fontsize=14)
    plt.tight_layout(); plt.show()

def iou(a, b):
    inter = np.logical_and(a, b).sum(); union = np.logical_or(a, b).sum()
    return inter / union if union else 0.0

## 3. Observação inicial (baseline): o que o SAM faz sozinho?

Primeiro, o gerador **automático** de máscaras (sem prompts). Ele segmenta *tudo* que encontra: céu, nuvens, prédios, fumaça. Isso mostra por que prompts são necessários.

In [ ]:
mask_generator = SamAutomaticMaskGenerator(sam, points_per_side=24, pred_iou_thresh=0.86, stability_score_thresh=0.9, min_mask_region_area=500)
auto_masks = mask_generator.generate(image)
auto_masks = sorted(auto_masks, key=lambda m: m["area"], reverse=True)
print("Máscaras automáticas encontradas:", len(auto_masks))

canvas = image.astype(np.float32).copy()
rng = np.random.default_rng(0)
for m in auto_masks:
    canvas[m["segmentation"]] = 0.45*canvas[m["segmentation"]] + 0.55*rng.integers(40, 255, 3)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].imshow(image); axs[0].set_title("Original")
axs[1].imshow(canvas.astype(np.uint8)); axs[1].set_title("SAM automático (sem prompts)")
for a in axs: a.axis("off")
plt.show()

## 4. Três configurações de prompt

**Defina abaixo** as coordenadas olhando a grade da imagem:
- `BOX = [x0, y0, x1, y1]`: caixa ao redor da pluma
- `POS`: pontos **dentro** da fumaça (rótulo 1), espalhados pela base, corpo e topo da pluma
- `NEG`: pontos em regiões que **não** são fumaça (rótulo 0): céu, nuvens vizinhas, prédios, chaminé

> Valores calibrados para a imagem `pluma.png`. Se usar outra imagem, ajuste olhando a grade.

In [ ]:
# Coordenadas definidas como fração da imagem (largura W, altura H), calibradas para a foto das chaminés
# (pluma cinza saindo da chaminé alta, em direção à direita, sobre céu nublado).
H, W = image.shape[:2]
def P(fx, fy): return (int(fx*W), int(fy*H))

BOX = np.array([int(0.245*W), int(0.21*H), int(0.995*W), int(0.62*H)])      # [x0, y0, x1, y1] ao redor da pluma
POS = [P(0.41, 0.46), P(0.555, 0.415), P(0.74, 0.40), P(0.89, 0.445)]       # dentro da fumaça (base, corpo, cauda)
NEG = [P(0.12, 0.185), P(0.62, 0.13),                                       # céu/nuvens acima
       P(0.49, 0.61),                                                       # nuvem logo abaixo da pluma
       P(0.24, 0.78), P(0.568, 0.905),                                      # chaminés
       P(0.80, 0.795)]                                                      # céu/nuvens inferiores

configs = {
    "A) Somente caixa":                        dict(box=BOX,  pos=[],  neg=[]),
    "B) Somente pontos positivos":             dict(box=None, pos=POS, neg=[]),
    "C) Caixa + positivos + negativos":        dict(box=BOX,  pos=POS, neg=NEG),
}

In [ ]:
predictor.set_image(image)   # calcula o embedding da imagem uma única vez

def run_sam(box=None, pos=(), neg=(), multimask=True, mask_input=None):
    pts = list(pos) + list(neg)
    coords = np.array(pts) if pts else None
    labels = np.array([1]*len(pos) + [0]*len(neg)) if pts else None
    masks, scores, logits = predictor.predict(
        point_coords=coords, point_labels=labels,
        box=None if box is None else np.array(box),
        mask_input=mask_input, multimask_output=multimask)
    return masks, scores, logits

results = {}
for name, cfg in configs.items():
    masks, scores, logits = run_sam(cfg["box"], cfg["pos"], cfg["neg"])
    best = int(np.argmax(scores))
    results[name] = dict(masks=masks, scores=scores, logits=logits, best=best, mask=masks[best], score=scores[best])
    side_by_side(image, masks[best], title=name, box=cfg["box"], pos=cfg["pos"], neg=cfg["neg"], score=scores[best])

#### As 3 hipóteses de máscara que o SAM devolve (multimask) para cada configuração
O SAM gera 3 máscaras (subparte / parte / objeto inteiro). Compare se a de maior score é realmente a melhor para a fumaça.

In [ ]:
for name, r in results.items():
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    for i in range(3):
        axs[i].imshow(overlay(image, r["masks"][i])); axs[i].axis("off")
        axs[i].set_title(f"{name}\nmáscara {i} | score={r['scores'][i]:.3f} | área={100*r['masks'][i].mean():.1f}%")
    plt.tight_layout(); plt.show()

## 5. Melhorias propostas

Problemas típicos: a máscara vaza para céu/nuvens/prédios, tem ilhas isoladas, buracos internos ou deixa de fora as bordas translúcidas da fumaça. Três melhorias:

1. **Pós-processamento morfológico**: abertura/fechamento, preenchimento de buracos e manter só o componente conectado que contém os pontos positivos.
2. **Refinamento iterativo**: reenviar os *logits* da primeira passada como `mask_input` do SAM.
3. **Pré-processamento de contraste (CLAHE)**: realçar a diferença entre fumaça e fundo antes de calcular o embedding.

In [ ]:
def postprocess(mask, pos_pts=(), open_k=5, close_k=15, min_area_frac=0.001):
    m = mask.astype(np.uint8)
    # 1) abertura remove pequenas ilhas/ruído; fechamento une partes da pluma
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k_open)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k_close)
    # 2) preencher buracos internos
    m = ndi.binary_fill_holes(m).astype(np.uint8)
    # 3) manter componentes que contêm pontos positivos (ou o maior, se não houver pontos)
    n, lab = cv2.connectedComponents(m)
    keep = np.zeros_like(m)
    ids = {lab[y, x] for (x, y) in pos_pts if 0 <= y < lab.shape[0] and 0 <= x < lab.shape[1] and lab[y, x] > 0}
    if not ids and n > 1:
        areas = [(lab == i).sum() for i in range(1, n)]
        ids = {1 + int(np.argmax(areas))}
    for i in ids: keep[lab == i] = 1
    return keep.astype(bool)

def refine_iterative(box, pos, neg, first_logits, n_iter=2):
    # reenvia o melhor logit (256x256) como entrada de máscara
    logit = first_logits
    for _ in range(n_iter):
        masks, scores, logits = run_sam(box, pos, neg, multimask=False, mask_input=logit[None, :, :])
        logit = logits[0]
    return masks[0], scores[0]

def enhance_contrast(img, clip=3.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

In [ ]:
improved = {}
for name, cfg in configs.items():
    r = results[name]
    # (1) pós-processamento
    m_post = postprocess(r["mask"], cfg["pos"])
    # (2) refinamento iterativo a partir do melhor logit
    m_ref, s_ref = refine_iterative(cfg["box"], cfg["pos"], cfg["neg"], r["logits"][r["best"]])
    m_ref_post = postprocess(m_ref, cfg["pos"])
    improved[name] = dict(post=m_post, refined=m_ref, refined_post=m_ref_post, score_ref=s_ref)
    side_by_side(image, m_ref_post, title=f"{name} - refinamento + pós-processamento",
                 box=cfg["box"], pos=cfg["pos"], neg=cfg["neg"], score=s_ref)

### Pré-processamento de contraste (CLAHE) na melhor configuração (C)

In [ ]:
image_clahe = enhance_contrast(image)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].imshow(image); axs[0].set_title("Original")
axs[1].imshow(image_clahe); axs[1].set_title("Após CLAHE")
for a in axs: a.axis("off")
plt.show()

cfg = configs["C) Caixa + positivos + negativos"]
predictor.set_image(image_clahe)
masks_c, scores_c, logits_c = run_sam(cfg["box"], cfg["pos"], cfg["neg"])
b = int(np.argmax(scores_c))
mask_clahe = postprocess(masks_c[b], cfg["pos"])
side_by_side(image, mask_clahe, title="C) com CLAHE + pós-processamento",
             box=cfg["box"], pos=cfg["pos"], neg=cfg["neg"], score=scores_c[b])
predictor.set_image(image)  # volta ao embedding da imagem original

## 6. Métricas comparativas
Sem *ground truth*, comparamos as configurações entre si (IoU) e medimos área e score. Se desejar, desenhe uma máscara de referência manual e informe em `GT_PATH`.

In [ ]:
GT_PATH = None   # ex.: "mascara_referencia.png" (branco = fumaça). Deixe None se não houver.

rows = []
ref_C = results["C) Caixa + positivos + negativos"]["mask"]
gt = None
if GT_PATH:
    gt = cv2.resize(cv2.imread(GT_PATH, 0), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST) > 127

for name in configs:
    r, im = results[name], improved[name]
    for tag, m in [("SAM bruto", r["mask"]), ("pós-proc.", im["post"]), ("refinado+pós", im["refined_post"])]:
        row = {"Configuração": name, "Versão": tag, "Área (%)": round(100*m.mean(), 2),
               "IoU vs C bruto": round(iou(m, ref_C), 3)}
        if gt is not None: row["IoU vs referência"] = round(iou(m, gt), 3)
        rows.append(row)
rows.append({"Configuração": "C) + CLAHE", "Versão": "CLAHE+pós", "Área (%)": round(100*mask_clahe.mean(), 2),
             "IoU vs C bruto": round(iou(mask_clahe, ref_C), 3),
             **({"IoU vs referência": round(iou(mask_clahe, gt), 3)} if gt is not None else {})})
df = pd.DataFrame(rows)
display(df)

## 7. Máscara final e exportação

In [ ]:
final_mask = improved["C) Caixa + positivos + negativos"]["refined_post"]   # troque pela melhor versão na sua análise
cfg = configs["C) Caixa + positivos + negativos"]

# Apresentação melhorada: máscara com transparência, contorno e recorte da pluma
fig, axs = plt.subplots(1, 4, figsize=(24, 6))
axs[0].imshow(image); axs[0].set_title("Original")
axs[1].imshow(final_mask, cmap="gray"); axs[1].set_title("Máscara binária final")
axs[2].imshow(overlay(image, final_mask, color=(255, 60, 60), alpha=0.45)); axs[2].set_title("Sobreposição + contorno")
cut = image.copy(); cut[~final_mask] = (cut[~final_mask] * 0.15).astype(np.uint8)
axs[3].imshow(cut); axs[3].set_title("Pluma destacada (fundo escurecido)")
for a in axs: a.axis("off")
plt.tight_layout(); plt.savefig("resultado_final.png", dpi=150); plt.show()

cv2.imwrite("mascara_final.png", final_mask.astype(np.uint8) * 255)
print("Salvos: mascara_final.png e resultado_final.png")
files.download("mascara_final.png"); files.download("resultado_final.png")

## 8. Análise crítica (preencha com base nos **seus** resultados)

### 8.1 Avaliação da máscara inicial do SAM
- **A máscara cobre a pluma?** *(descreva: cobre o corpo principal? o topo? a base?)*
- **Inclui regiões que não pertencem a ela?** *(céu, nuvens, edifícios, chaminé, reflexos? cite a configuração em que isso ocorreu)*
- **Deixa partes da fumaça de fora?** *(bordas translúcidas, filamentos finos, topo difuso)*

### 8.2 Comparação das configurações de prompt
| Configuração | Pontos fortes | Pontos fracos |
|---|---|---|
| A) Somente caixa | Delimita a região de interesse sem exigir cliques precisos | A caixa também contém céu/nuvens e o SAM pode segmentar o fundo ou o objeto dominante da caixa |
| B) Somente pontos positivos | Direciona o SAM ao interior da fumaça | Sem limite espacial nem exemplos de "não fumaça", a máscara tende a vazar para nuvens de textura parecida |
| C) Caixa + positivos + negativos | Combina limite espacial e exclusão explícita de nuvens/céu/prédios | Exige mais esforço manual e escolha cuidadosa dos pontos negativos |

*(Confirme ou corrija a tabela com o que você observou; cite as áreas e IoU da tabela de métricas.)*

### 8.3 Características da imagem que dificultaram a segmentação
- **Iluminação:** *(contraluz, sol por trás da pluma, sombras, zonas super ou subexpostas; a fumaça iluminada de lados diferentes tem tons distintos e o SAM pode dividi-la em partes)*
- **Contraste:** *(fumaça clara contra céu claro ou fumaça cinza contra fundo cinza gera bordas fracas, e o SAM depende fortemente de bordas)*
- **Semelhança entre fumaça e nuvens:** *(cor, textura e opacidade parecidas; sem bordas nítidas, a máscara tende a se fundir com nuvens adjacentes)*
- **Natureza difusa/translúcida da fumaça:** *(gradiente de opacidade; não há fronteira bem definida, então qualquer máscara binária é uma simplificação)*
- **Outros:** *(edifícios, chaminés, linhas de energia, neblina, resolução)*

### 8.4 Efeito das melhorias
- **Pós-processamento:** *(removeu ilhas? preencheu buracos? por quanto mudou a área?)*
- **Refinamento iterativo:** *(a borda ficou mais coerente?)*
- **CLAHE:** *(melhorou ou piorou? em que região? o realce de contraste também realça nuvens e ruído, podendo piorar o resultado)*

### 8.5 Limitações e trabalhos futuros
- O SAM não "sabe" o que é fumaça: segmenta objetos genéricos e depende dos prompts.
- Possíveis extensões: modelos com texto (Grounded-SAM / GroundingDINO com o prompt *"smoke plume"*), ajuste fino (fine-tuning) em dados de fumaça, uso de modelos maiores (`vit_h`) e máscara "suave" por limiar de opacidade/cor dentro da região.